In [1]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("S10-Dataset-Peru-Unificado")
    .master("local[2]")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

RUTA = "/workspace/datasets/historicos/peru_dias/*.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(RUTA)
)

print("REGISTROS:", df.count())
print("COLUMNAS:", df.columns)
df.printSchema()
df.show(5, truncate=False)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/06 16:12:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/06 16:12:42 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/11/06 16:12:52 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /workspace/datasets/historicos/peru_dias/*.csv.
java.io.FileNotFoundException: File /workspace/datasets/historicos/peru_dias/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.ja

REGISTROS: 184766
COLUMNAS: ['icao24', 'eventTimestampUTC', 'epochSeconds', 'latitude', 'longitude', 'baroAltitude_m', 'geoAltitude_m', 'velocity_mps', 'trueTrack_deg', 'verticalRate_mps', 'geoVerticalRate_mps', 'onGround', 'positionSource', 'positionStale', 'newLeg', 'altitudeIsGeometric', 'verticalRateIsGeometric', 'callsign', 'aircraftType', 'sourceDataset', 'sourceFile']
root
 |-- icao24: string (nullable = true)
 |-- eventTimestampUTC: timestamp (nullable = true)
 |-- epochSeconds: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- baroAltitude_m: double (nullable = true)
 |-- geoAltitude_m: double (nullable = true)
 |-- velocity_mps: double (nullable = true)
 |-- trueTrack_deg: double (nullable = true)
 |-- verticalRate_mps: double (nullable = true)
 |-- geoVerticalRate_mps: double (nullable = true)
 |-- onGround: boolean (nullable = true)
 |-- positionSource: string (nullable = true)
 |-- positionStale: boolean (nullabl

In [2]:
# Usar epochSeconds como referencia UTC si existe.
if "epochSeconds" in df.columns:
    df = df.withColumn(
        "fechaHora",
        F.timestamp_seconds(F.col("epochSeconds").cast("long"))
    )
else:
    df = df.withColumn(
        "fechaHora",
        F.to_timestamp("eventTimestampUTC")
    )

resumen = (
    df.withColumn("fecha", F.to_date("fechaHora"))
    .groupBy("fecha")
    .agg(
        F.count("*").alias("observaciones"),
        F.countDistinct("icao24").alias("aeronaves")
    )
    .orderBy("fecha")
)

resumen.show(10, False)

print("TOTAL:", df.count())
print("AERONAVES DISTINTAS:", df.select("icao24").distinct().count())

+----------+-------------+---------+
|fecha     |observaciones|aeronaves|
+----------+-------------+---------+
|2024-12-26|35963        |160      |
|2024-12-27|40252        |169      |
|2024-12-28|37777        |170      |
|2024-12-29|38154        |168      |
|2024-12-30|32620        |171      |
+----------+-------------+---------+

TOTAL: 184766


[Stage 15:======================================>                   (2 + 1) / 3]

AERONAVES DISTINTAS: 401


In [5]:
from pyspark.sql import functions as F
from datetime import datetime, timezone

# Mantener el dataset original de ADSB.lol sin alterarlo
# dataset ya contiene eventTimestamp con fechas de 2024

RUTA_REPLAY = "/workspace/artifacts/s10/peru_replay_noviembre_2026"

# 1. Obtener el instante final real del dataset
fecha_final_original = dataset.agg(
    F.max("eventTimestamp")
).first()[0]

# 2. Establecer fin de reproducción:
# 6 de noviembre de 2026, 16:20:12 UTC
# (hora del replay que ya publicamos a Kafka)
fecha_final_replay = datetime(
    2026, 11, 6, 16, 20, 12, tzinfo=timezone.utc
)

# 3. Calcular el desplazamiento en segundos
original_utc = fecha_final_original.replace(
    tzinfo=timezone.utc
)

desplazamiento = int(
    fecha_final_replay.timestamp()
    - original_utc.timestamp()
)

# 4. Crear un nuevo timestamp de reproducción
dataset_replay = (
    dataset
    .withColumn(
        "originalEventTimestamp",
        F.col("eventTimestamp")
    )
    .withColumn(
        "eventTimestamp",
        F.timestamp_seconds(
            F.unix_timestamp("originalEventTimestamp")
            + F.lit(desplazamiento)
        )
    )
    .withColumn(
        "fecha",
        F.to_date("eventTimestamp")
    )
    .withColumn(
        "historicalReplay",
        F.lit(True)
    )
)

# 5. Verificar ambas fechas
dataset_replay.select(
    F.min("originalEventTimestamp").alias("inicio_original"),
    F.max("originalEventTimestamp").alias("fin_original"),
    F.min("eventTimestamp").alias("inicio_replay"),
    F.max("eventTimestamp").alias("fin_replay")
).show(truncate=False)

dataset_replay.groupBy("fecha") \
    .count() \
    .orderBy("fecha") \
    .show(10, False)

# 6. Guardar en una ruta separada
dataset_replay.write \
    .mode("overwrite") \
    .partitionBy("fecha") \
    .parquet(RUTA_REPLAY)

print("PARQUET REPLAY CREADO:", RUTA_REPLAY)

+-------------------+-------------------+-------------------+-------------------+
|inicio_original    |fin_original       |inicio_replay      |fin_replay         |
+-------------------+-------------------+-------------------+-------------------+
|2024-12-26 00:00:03|2024-12-30 23:57:34|2026-11-01 16:22:41|2026-11-06 16:20:12|
+-------------------+-------------------+-------------------+-------------------+



+----------+-----+
|fecha     |count|
+----------+-----+
|2026-11-01|13200|
|2026-11-02|36698|
|2026-11-03|39994|
|2026-11-04|38128|
|2026-11-05|35384|
|2026-11-06|21349|
+----------+-----+



PARQUET REPLAY CREADO: /workspace/artifacts/s10/peru_replay_noviembre_2026
